# 03 - Red neuronal informada por fisica

Incorporamos conocimiento fisico en el entrenamiento. Para un proyectil ideal se cumple `d²y/dt² = -g`. Entrenamos una red `y_theta(t)` y un parametro aprendible `g`.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
raw = np.genfromtxt(ROOT / 'data' / 'trajectory_extracted.csv', delimiter=',', names=True)
t = raw['t_s'].astype('float32')
y = raw['y_m'].astype('float32')
t_tensor = tf.convert_to_tensor(t.reshape(-1,1), dtype=tf.float32)
y_tensor = tf.convert_to_tensor(y.reshape(-1,1), dtype=tf.float32)


## Modelo y perdida

La perdida total combina error contra datos y residuo fisico `d²y/dt² + g`.


In [ ]:
keras.utils.set_random_seed(0)
model = keras.Sequential([
    layers.Input(shape=(1,)),
    layers.Dense(32, activation='tanh'),
    layers.Dense(32, activation='tanh'),
    layers.Dense(1)
])
g_param = tf.Variable([9.0], dtype=tf.float32, name='g_param')
optimizer = optimizers.Adam(learning_rate=0.01)
_ = model(t_tensor)

def segunda_derivada_modelo(t_in, training=True):
    with tf.GradientTape() as tape2:
        tape2.watch(t_in)
        with tf.GradientTape() as tape1:
            tape1.watch(t_in)
            y_out = model(t_in, training=training)
        dy_dt = tape1.gradient(y_out, t_in)
    d2y_dt2 = tape2.gradient(dy_dt, t_in)
    return y_out, d2y_dt2

def train(num_epochs=3000, alpha=1.0):
    for epoch in range(num_epochs):
        with tf.GradientTape() as tape:
            y_pred = model(t_tensor, training=True)
            loss_data = tf.reduce_mean(tf.square(y_pred - y_tensor))
            _, d2y_dt2 = segunda_derivada_modelo(t_tensor, training=True)
            loss_phys = tf.reduce_mean(tf.square(d2y_dt2 + g_param))
            loss = loss_data + alpha * loss_phys
        variables = model.trainable_variables + [g_param]
        grads = tape.gradient(loss, variables)
        optimizer.apply_gradients(zip(grads, variables))
        if epoch % 600 == 0:
            print(f'Epoca {epoch:4d} - datos: {loss_data.numpy():.3e} - fisica: {loss_phys.numpy():.3e} - g: {g_param.numpy()[0]:.4f}')


In [ ]:
train(num_epochs=3000, alpha=1.0)


## Evaluacion

Comparamos la trayectoria aprendida, el valor de `g` y el residuo fisico.


In [ ]:
y_pred = model(t_tensor, training=False).numpy().squeeze()
g_learned = float(g_param.numpy()[0])
_, d2y_dt2 = segunda_derivada_modelo(t_tensor, training=False)
residuo = (d2y_dt2 + g_param).numpy().squeeze()
print(f'g aprendido: {g_learned:.4f} m/s^2')
print(f'Error medio residuo: {np.mean(residuo**2):.4e}')

plt.figure(figsize=(8,4))
plt.plot(t, y, 'o', label='datos')
plt.plot(t, y_pred, '-', label='red informada')
plt.xlabel('t [s]'); plt.ylabel('y [m]')
plt.grid(True, alpha=0.3); plt.legend(); plt.show()

plt.figure(figsize=(6,4))
plt.plot(t, residuo, '-o')
plt.axhline(0, color='k', linestyle='--')
plt.xlabel('t [s]'); plt.ylabel('d²y/dt² + g')
plt.grid(True, alpha=0.3); plt.show()


## Reflexiones

- La perdida fisica reduce libertad, pero mejora interpretabilidad.
- El parametro `g` aprendido conecta la red con una magnitud fisica.
- La misma idea puede adaptarse a pendulos, orbitas o sistemas con roce.
